In [39]:
# import necessary dependencies
import os
from openai import AsyncOpenAI, OpenAI
import requests
from pydantic import BaseModel, Field
import asyncio
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, guardrail, set_tracing_disabled, set_tracing_export_api_key, trace, function_tool, SQLiteSession
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel

# load env variables
load_dotenv()

set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))

# 2. Directly create the client using your GEMINI_API_KEY from .env
gemini_client = AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 3. Pass the client to the model adapter
gemini_model = OpenAIChatCompletionsModel(
    openai_client=gemini_client,
    model="gemini-2.5-flash"
)

In [2]:
instructions = """
You are a sales agent working for ComplAI,
a company thatprovides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write compelling sales emails that are likely to get a response.
"""

In [5]:
sales_agent1 = Agent(name="sales-agent-1", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="sales-agent-2", instructions=instructions, model=gemini_model)
sales_agent3 = Agent(name="sales-agent-3", instructions=instructions, model=gemini_model)

In [6]:
description = "Use this tool to write a sales email. In the input , just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [7]:
import os
import asyncio
import smtplib
import requests
from email.message import EmailMessage
from typing import Dict

from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent

# 1. FIXED: Removed OpenAIChatCompletionsModel from main agents import
from agents import Agent, Runner, set_tracing_export_api_key, trace, function_tool, ModelSettings

# 2. FIXED: Imported OpenAIChatCompletionsModel from its submodule
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")

# 3. FIXED: Corrected spelling to match standard EMAIL_APP_PASSWORD
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject} \n\n{text_body}")

In [8]:
USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [11]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """

    send_message(subject, text_body, html_body)
    return "Email sent succesfully"

In [12]:
tools = [tool1, tool2, tool3, send_email_tool]

In [17]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready,  one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is the most effective.

3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=gemini_model)



In [18]:
with trace("Sales Manager across different models"):
    result = await Runner.run(sales_manager, task)
print(result)

RunResult:
- Last agent: Agent(name="Sales Manager", ...)
- Final output (str):
    I have generated three email drafts, evaluated them, and selected the most effective one. I have also used the `send_email_tool` to send the chosen email.
- 9 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [25]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [27]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [28]:
email = """
Hi [first_name],

I'm hitting you up to see if you's like to buy our product. It's really great. You'll miss out if you don't buy it.

Laters.

Sage
"""

In [29]:
checker = Agent(name="Checker", instructions="You review potential sales emails", model=gemini_model, output_type=EmailReview)
result = await Runner.run(checker, email)

In [30]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [31]:
review.is_professional

False

In [ ]:
from agents import Agent, Runner, output_guardrail, GuardrailFunctionOutput
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

In [45]:
cowboy_instructions = instructions + "\nspeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [47]:
try:
    result = await Runner.run(sales_agent_cowboy, "write a cold sales email")
    print(result.final_output)
except Exception as e:
    print(f"Guardrail blocked the message: {e}")

Guardrail blocked the message: Guardrail OutputGuardrail triggered tripwire


In [48]:
simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
result = await Runner.run(simple_cowboy, "Write a cold sales email")
email = result.final_output
print(email)

Howdy Partner! This ain't my first rodeo, and I know a good piece of prospecting when I see it.

Alright, saddle up, because finding that *single best* cold sales email is like tracking the prize bull – gotta be precise, gotta hit 'em where it counts.

I'm gonna use our `sales_writer` tool, tell it exactly what we need, and then I'll put my cowboy hat on straight to pick the champ.

**My instructions for the `sales_writer` tool:**

*   **Company:** ComplAI – we're an AI-powered compliance automation solution.
*   **Target Audience:** Busy executives (CFOs, COOs, Compliance Officers) in mid-to-large businesses.
*   **Problem:** Manual, time-consuming, error-prone, and risky compliance processes.
*   **Solution:** ComplAI automates, reduces risk, saves time/cost, ensures accuracy.
*   **Goal:** Book a quick 15-minute discovery call to see if ComplAI can lighten their load.
*   **Tone:** Friendly, direct, professional, with a touch of that ol' frontier spirit (like yours truly).
*   **Key

In [49]:
result = await Runner.run(checker, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")


The email is not professional or has placeholders and will not be sent
